# Viewer: read existing records and verify denied changes
Run the Markdown setup first. Open this notebook beside private-lab.json in a fresh Python kernel. Run cells in order. This uses isolated run-specific Polaris objects.


In [ ]:
import json, os, socket
from pathlib import Path
from pyspark.sql import SparkSession
config = json.loads(Path('private-lab.json').read_text())
assert config['catalog'].startswith('polaris_ops_')
ROLE = 'viewer'
identity = config['principals'][ROLE]
assert SparkSession.getActiveSession() is None, 'Restart the kernel to switch identities safely'
builder = (SparkSession.builder.appName('polaris-' + ROLE + '-' + config['run_id'])
    .master(config['spark_master'])
    .config('spark.driver.host', os.environ.get('POD_IP') or socket.gethostbyname(socket.gethostname()))
    .config('spark.driver.bindAddress', '0.0.0.0')
    .config('spark.cores.max', '1')
    .config('spark.executor.cores', '1')
    .config('spark.executor.memory', '512m')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.redaction.regex', '(?i)secret|password|token|access.?key|credential')
    .config('spark.sql.catalog.lab', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.lab.type', 'rest')
    .config('spark.sql.catalog.lab.uri', config['polaris_uri'])
    .config('spark.sql.catalog.lab.warehouse', config['catalog'])
    .config('spark.sql.catalog.lab.credential', identity['clientId'] + ':' + identity['clientSecret'])
    .config('spark.sql.catalog.lab.scope', 'PRINCIPAL_ROLE:ALL')
    .config('spark.sql.catalog.lab.header.Polaris-Realm', 'POLARIS')
    .config('spark.sql.catalog.lab.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.lab.s3.endpoint', config['s3_endpoint'])
    .config('spark.sql.catalog.lab.s3.path-style-access', 'true')
    .config('spark.sql.catalog.lab.s3.access-key-id', config['s3_access_key'])
    .config('spark.sql.catalog.lab.s3.secret-access-key', config['s3_secret_key'])
    .config('spark.sql.catalog.lab.client.region', 'us-east-1')
    .config('spark.hadoop.fs.s3a.endpoint', config['s3_endpoint'])
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.access.key', config['s3_access_key'])
    .config('spark.hadoop.fs.s3a.secret.key', config['s3_secret_key']))
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.master == config['spark_master']
print('Role:', ROLE, 'Catalog:', config['catalog'], 'Application:', spark.sparkContext.applicationId)
# Never display identity, config or the full SparkConf: they contain credentials.


In [ ]:
expected = [(1,'north',120),(2,'north',100),(3,'south',50),(4,'south',150)]
actual = [tuple(r) for r in spark.table('lab.sales.orders').orderBy('order_id').collect()]
assert actual == expected, actual
assert spark.table('lab.sales.orders').groupBy().sum('amount').first()[0] == 420
assert [tuple(r) for r in spark.table('lab.sales.order_totals').orderBy('region').collect()] == [('north',220),('south',200)]
spark.table('lab.sales.orders').orderBy('order_id').show()
print('PASS: viewer read exact rows and total 420')


In [ ]:
def expect_authorization_denied(statement):
    try:
        spark.sql(statement).collect()
    except Exception as error:
        message = str(error).lower()
        assert any(word in message for word in ['forbidden', 'not authorized', '403', 'accessdenied', 'permission denied']), type(error).__name__
        print('PASS: authorization denied:', statement)
    else:
        raise AssertionError('Unexpected permission: ' + statement)
expect_authorization_denied("ALTER TABLE lab.sales.orders SET TBLPROPERTIES ('viewer_changed'='yes')")
expect_authorization_denied('CREATE NAMESPACE lab.viewer_should_not_create')
properties = {r.key:r.value for r in spark.sql('SHOW TBLPROPERTIES lab.sales.orders').collect()}
assert 'viewer_changed' not in properties
assert spark.table('lab.sales.orders').count() == 4
print('PASS: viewer could not modify table metadata or create a namespace')


In [ ]:
spark.stop()
print('VIEWER_NOTEBOOK_PASS')
